In [1]:
import sqlite3
import pandas as pd
import numpy as np
import os

In [2]:
import numpy as np
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
from sklearn.model_selection import train_test_split

In [3]:
static_cols = [
    'grade','sub_grade']
dynamic_cols = [
    'default_indicator_t1','default_indicator_t2','default_indicator_t3', 'default_indicator_t4',
    'fico_score_t1','fico_score_t2','fico_score_t3','fico_score_t4'
]

In [5]:
# Read your CSV file. It should contain all columns from static_cols and dynamic_cols.
data = pd.read_csv("bnpl_loans_with_age_change.csv")

data['fico_score_t1'] = pd.cut(data['fico_score_t1'], bins=10, labels=False)
data['fico_score_t2'] = pd.cut(data['fico_score_t2'], bins=10, labels=False)
data['fico_score_t3'] = pd.cut(data['fico_score_t3'], bins=10, labels=False)
data['fico_score_t4'] = pd.cut(data['fico_score_t4'], bins=10, labels=False)
data['fico_score_t5'] = pd.cut(data['fico_score_t5'], bins=10, labels=False)

# (a) Process static columns:
static_df = data[static_cols].copy()
# Assign the static columns to time period 0. We build a MultiIndex from the plain list.
static_df.columns = pd.MultiIndex.from_product([static_df.columns.tolist(), [0]])

# (b) Process dynamic columns:
# The raw dynamic column names are assumed to be labeled with suffixes _t1 ... _t6.
# We remap _t1 -> time 0, _t2 -> time 1, …, _t6 -> time 5.
time_slices = [str(i) for i in range(1, 7)]  # raw suffixes as strings ("1".."6")
dynamic_data = {}
for col in dynamic_cols:
    try:
        var, t = col.rsplit('_t', 1)
    except ValueError:
        continue
    if t in time_slices:
        dynamic_data.setdefault(int(t), {})[var] = data[col]

dynamic_dfs = {}
for t in sorted(dynamic_data.keys()):
    df_t = pd.DataFrame(dynamic_data[t])
    # Build MultiIndex columns with time equal to t-1 (so that _t1 becomes period 0, ..., _t6 becomes period 5)
    df_t.columns = pd.MultiIndex.from_product([df_t.columns, [t-1]])
    dynamic_dfs[t-1] = df_t

# (c) Combine static and dynamic data:
# At time period 0, combine static_df with dynamic slice 0.
combined_df = static_df.copy()
for t in sorted(dynamic_dfs.keys()):
    combined_df = combined_df.join(dynamic_dfs[t])

# Optional: Check that the combined DataFrame has columns in the format (variable, time)
print("Combined DataFrame columns:")
print(combined_df.columns.tolist())


/var/folders/bm/0wq19v017y79n8fk7_72tt4c0000gn/T/ipykernel_24520/3561558918.py:2: DtypeWarning: Columns (19,49,59,118,129,130,131,134,135,136,139) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("bnpl_loans_with_age_change.csv")


Combined DataFrame columns:
[('grade', 0), ('sub_grade', 0), ('default_indicator', 0), ('fico_score', 0), ('default_indicator', 1), ('fico_score', 1), ('default_indicator', 2), ('fico_score', 2), ('default_indicator', 3), ('fico_score', 3)]


In [6]:
# -----------------------------------------------------------------------------
# 4. Define the Unroll Function for the Credit Risk DBN
# -----------------------------------------------------------------------------
def unroll_credit_risk_dbn(num_time_slices):
    """
    Unrolls a credit risk dynamic Bayesian network over num_time_slices time periods (0 to num_time_slices-1).

    Static variables (from static_cols) are available only at time period 0.
    Dynamic variables (derived by removing the '_tX' suffix from dynamic_cols) are available in every time period.
    
    The dependency structure in this example is:
      - At time period 0:
            For each static variable s, add an edge: (s, 0) -> (default_indicator, 0)
            Also, if 'fico_score' is among the dynamic variables, add: (fico_score, 0) -> (default_indicator, 0)
      - For time periods 1 to num_time_slices-1:
            For every dynamic variable d, add a transition edge: (d, t-1) -> (d, t)
            Also, add an intra-slice edge: (fico_score, t) -> (default_indicator, t)
            And a transition edge: (default_indicator, t-1) -> (default_indicator, t)
    """
    edges = []
    # Build dynamic variable names by stripping the suffix from dynamic_cols.
    dynamic_vars = list({col.rsplit('_t', 1)[0] for col in dynamic_cols})
    
    # Time period 0 edges:
    for s in static_cols:
        edges.append(((s, 0), ('default_indicator', 0)))
    if 'fico_score' in dynamic_vars:
        edges.append((('fico_score', 0), ('default_indicator', 0)))
        
    # For time periods 1 to num_time_slices-1:
    for t in range(1, num_time_slices):
        # Transition edges for each dynamic variable.
        for d in dynamic_vars:
            edges.append(((d, t-1), (d, t)))
        # Intra-slice edge from fico_score to default_indicator.
        if 'fico_score' in dynamic_vars:
            edges.append((('fico_score', t), ('default_indicator', t)))
        # Transition edge for default_indicator.
        edges.append((('default_indicator', t-1), ('default_indicator', t)))
        
    model = DiscreteBayesianNetwork(edges)
    return model


In [7]:
# -----------------------------------------------------------------------------
# 5. Train the Unrolled DBN on the Data
# -----------------------------------------------------------------------------
num_time_slices = 4  # Time periods 0 through 5.
model = unroll_credit_risk_dbn(num_time_slices)

# Split the data into training and testing sets (e.g., 80% training, 20% testing)
train_data, test_data = train_test_split(combined_df, test_size=0.2, random_state=42)

model.fit(train_data, estimator=MaximumLikelihoodEstimator)

print("\nTraining complete. Learned CPDs:")
for cpd in model.get_cpds():
    print(cpd)

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {('grade', 0): 'C', ('sub_grade', 0): 'C', ('default_indicator', 0): 'N', ('fico_score', 0): 'N', ('default_indicator', 1): 'N', ('fico_score', 1): 'N', ('default_indicator', 2): 'N', ('fico_score', 2): 'N', ('default_indicator', 3): 'N', ('fico_score', 3): 'N'}



Training complete. Learned CPDs:
+-----------------+------------+
| ('grade', 0)(A) | 0.158697   |
+-----------------+------------+
| ('grade', 0)(B) | 0.311293   |
+-----------------+------------+
| ('grade', 0)(C) | 0.313411   |
+-----------------+------------+
| ('grade', 0)(D) | 0.153411   |
+-----------------+------------+
| ('grade', 0)(E) | 0.048823   |
+-----------------+------------+
| ('grade', 0)(F) | 0.0124178  |
+-----------------+------------+
| ('grade', 0)(G) | 0.00194749 |
+-----------------+------------+
+-------------------------------+-----+----------------------+
| ('fico_score', 0)             | ... | ('fico_score', 0)(9) |
+-------------------------------+-----+----------------------+
| ('grade', 0)                  | ... | ('grade', 0)(G)      |
+-------------------------------+-----+----------------------+
| ('sub_grade', 0)              | ... | ('sub_grade', 0)(G5) |
+-------------------------------+-----+----------------------+
| ('default_indicator', 0)(0.0

In [5]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.estimators import MaximumLikelihoodEstimator

# Define the columns (features) as provided
cols = [
    "loan_amnt_disc",
    #"int_rate_disc",
    "installment_disc",
    #"pub_rec_disc",
    #"revol_bal_disc",
    #"revol_util_disc",
    "total_acc_disc",
    #"annual_inc_disc",
    #"delinq_2yrs_disc",
    #"inq_last_6mths_disc",
    #"open_acc_disc",
    "gdp_growth_t1_disc",
    "unemployment_rate_t1_disc",
    "fedfunds_t1_disc",
    "inflation_t1_disc",
    "housing_price_t1_disc",
    #"grade",
    #"sub_grade",
    "emp_length",
    "home_ownership",
    #"purpose",
    "fico_bucket_pct",
    "default_indicator",
    "age_group",
    #"issue_quarter",
]

# Load your pre-discretized data
data = pd.read_csv('rebalanced_age_data_discretized.csv')

# Filter the data to include only the relevant columns
data = data[cols]

# Define a plausible structure for predicting default.
# Most features have a direct influence on "default_indicator".
# An example structure: loan characteristics, borrower factors, and macroeconomic factors all point to default.
# Additionally, "grade" is assumed to influence "sub_grade", and sub_grade influences default.
edges = [
    # Loan Characteristics Group
    ("loan_amnt_disc", "installment_disc"),
    
    # Borrower Factors Group
    ("emp_length", "home_ownership"),
    ("home_ownership", "fico_bucket_pct"),
    ("fico_bucket_pct", "age_group"),
    
    # Macroeconomic Conditions Group
    ("gdp_growth_t1_disc", "unemployment_rate_t1_disc"),
    ("unemployment_rate_t1_disc", "fedfunds_t1_disc"),
    ("fedfunds_t1_disc", "inflation_t1_disc"),
    ("inflation_t1_disc", "housing_price_t1_disc"),
    
    # Final arrows to the target variable.
    ("installment_disc", "default_indicator"),
    ("age_group", "default_indicator"),
    ("housing_price_t1_disc", "default_indicator")
]

# Initialize the Bayesian network with the assumed structure.
model = DiscreteBayesianNetwork(edges)

# Fit the parameters for the network using Bayesian estimation.
# The BayesianEstimator can be tuned with different priors if desired.
#model.fit(data, estimator=BayesianEstimator, prior_type='BDeu')


In [6]:
#model.fit(data, estimator=MaximumLikelihoodEstimator)
model.fit(data, estimator=BayesianEstimator, prior_type='BDeu')
# Optionally, inspect the conditional probability distributions (CPDs) to validate the learned parameters.
print("Learned CPDs:")
for cpd in model.get_cpds():
    print(cpd)

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'loan_amnt_disc': 'N', 'installment_disc': 'N', 'total_acc_disc': 'N', 'gdp_growth_t1_disc': 'N', 'unemployment_rate_t1_disc': 'N', 'fedfunds_t1_disc': 'N', 'inflation_t1_disc': 'N', 'housing_price_t1_disc': 'N', 'emp_length': 'C', 'home_ownership': 'C', 'fico_bucket_pct': 'C', 'default_indicator': 'N', 'age_group': 'C'}


Learned CPDs:
+---------------------+----------+
| loan_amnt_disc(0.0) | 0.171126 |
+---------------------+----------+
| loan_amnt_disc(1.0) | 0.305178 |
+---------------------+----------+
| loan_amnt_disc(2.0) | 0.523696 |
+---------------------+----------+
+-----------------------+-----+-----------------------+
| loan_amnt_disc        | ... | loan_amnt_disc(2.0)   |
+-----------------------+-----+-----------------------+
| installment_disc(0.0) | ... | 9.922008399095482e-05 |
+-----------------------+-----+-----------------------+
| installment_disc(1.0) | ... | 0.17649406986939867   |
+-----------------------+-----+-----------------------+
| installment_disc(2.0) | ... | 0.8234067100466104    |
+-----------------------+-----+-----------------------+
+------------------------+-----+----------------------------+
| age_group              | ... | age_group(65+)             |
+------------------------+-----+----------------------------+
| housing_price_t1_disc  | ... | housing_price_t1_d

In [ ]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.estimators import MaximumLikelihoodEstimator

# Define the columns (features) as provided
cols = [
    #"loan_amnt_disc",
    #"int_rate_disc",
    #"installment_disc",
    #"pub_rec_disc",
    #"revol_bal_disc",
    #"revol_util_disc",
    #"total_acc_disc",
    #"annual_inc_disc",
    #"delinq_2yrs_disc",
    #"inq_last_6mths_disc",
    #"open_acc_disc",
    #"gdp_growth_t1_disc",
    #"unemployment_rate_t1_disc",
    #"fedfunds_t1_disc",
    #"inflation_t1_disc",
    #"housing_price_t1_disc",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "purpose",
    "fico_bucket_pct",
    "default_indicator",
    "age_group",
    #"issue_quarter",
]

# Load your pre-discretized data
data = pd.read_csv('rebalanced_age_data_discretized.csv')

# Filter the data to include only the relevant columns
data = data[cols]

# Define a plausible structure for predicting default.
# Most features have a direct influence on "default_indicator".
# An example structure: loan characteristics, borrower factors, and macroeconomic factors all point to default.
# Additionally, "grade" is assumed to influence "sub_grade", and sub_grade influences default.
edges = [
    ("grade","default_indicator"),
    ("sub_grade","default_indicator"),
    ("emp_length","default_indicator"),
    ("home_ownership","default_indicator"),
    ("purpose","default_indicator"),
    ("fico_bucket_pct","default_indicator"),
    ("age_group","default_indicator")
]

# Initialize the Bayesian network with the assumed structure.
model = DiscreteBayesianNetwork(edges)

# Fit the parameters for the network using Bayesian estimation.
# The BayesianEstimator can be tuned with different priors if desired.
#model.fit(data, estimator=BayesianEstimator, prior_type='BDeu')

#model.fit(data, estimator=MaximumLikelihoodEstimator)
model.fit(data, estimator=BayesianEstimator, prior_type='BDeu')
# Optionally, inspect the conditional probability distributions (CPDs) to validate the learned parameters.
print("Learned CPDs:")
for cpd in model.get_cpds():
    print(cpd)

In [9]:
# Create a directory to store the CPD CSV files
import os
import pandas as pd
import numpy as np
from itertools import product
from pgmpy.factors.discrete import TabularCPD
output_dir = "cpd_tables"
os.makedirs(output_dir, exist_ok=True)

def save_cpd_as_csv(cpd):
    """
    Convert a TabularCPD into a pandas DataFrame and save it as CSV.
    We assume that the CPD has an attribute state_names, which is a dictionary 
    mapping variable and evidence variables to lists of state names.
    """
    # Check if there is evidence (parents)
    if not cpd.evidence:
        # No evidence: CPD is just a column vector.
        # Use the state names of the variable if provided; otherwise, use indices.
        if cpd.state_names and cpd.variable in cpd.state_names:
            row_labels = cpd.state_names[cpd.variable]
        else:
            row_labels = [f"{cpd.variable}_state_{i}" for i in range(cpd.variable_card)]
        # cpd.values should have shape (variable_card, 1)
        df = pd.DataFrame(cpd.values, index=row_labels, columns=["Probability"])
    else:
        # With evidence: columns correspond to combinations of evidence states.
        # Get the state names for each evidence variable. If not provided, use generic names.
        evidence_state_lists = []
        for ev in cpd.evidence:
            if cpd.state_names and ev in cpd.state_names:
                evidence_state_lists.append(cpd.state_names[ev])
            else:
                # Create default state names
                evidence_state_lists.append([f"{ev}_state_{i}" for i in range(cpd.evidence_card[cpd.evidence.index(ev)])])
        
        # Build multi-index for the columns using the Cartesian product of evidence states.
        col_labels = list(product(*evidence_state_lists))
        # The target variable's row labels.
        if cpd.state_names and cpd.variable in cpd.state_names:
            row_labels = cpd.state_names[cpd.variable]
        else:
            row_labels = [f"{cpd.variable}_state_{i}" for i in range(cpd.variable_card)]
        # cpd.values should have shape (variable_card, prod(evidence_card))
        df = pd.DataFrame(cpd.values, index=row_labels, columns=pd.MultiIndex.from_tuples(col_labels, names=cpd.evidence))
    # Save the DataFrame to a CSV file.
    filename = os.path.join(output_dir, f"cpd_{cpd.variable}.csv")
    df.to_csv(filename)
    print(f"Saved CPD for {cpd.variable} to {filename}")

def save_cpd_as_text(cpd):
    """Simply write the string representation of the CPD to a text file."""
    filename = os.path.join(output_dir, f"cpd_{cpd.variable}.txt")
    with open(filename, "w") as f:
        f.write(str(cpd))
    print(f"Saved CPD for {cpd.variable} as text to {filename}")

# After model.fit(), iterate over all CPDs
print("Learned CPDs:")
for cpd in model.get_cpds():
    print(cpd)
    
    # Option 1: Save as CSV table
    save_cpd_as_csv(cpd)
    
    # Option 2: Save as text (optional, if you want both)
    save_cpd_as_text(cpd)

Learned CPDs:
+---------------------+----------+
| loan_amnt_disc(0.0) | 0.171126 |
+---------------------+----------+
| loan_amnt_disc(1.0) | 0.305178 |
+---------------------+----------+
| loan_amnt_disc(2.0) | 0.523696 |
+---------------------+----------+


AttributeError: 'TabularCPD' object has no attribute 'evidence'

In [8]:
# -----------------------------------------------------------------------------
# 6. Inference Example
# -----------------------------------------------------------------------------
infer = VariableElimination(model)

# Select a test instance from train_data at time period 0.
test_instance = train_data.iloc[0]
# Gather evidence from time period 0: both static and dynamic.
evidence = {}
for s in static_cols:
    evidence[(s, 0)] = int(test_instance[(s, 0)])
# Dynamic variables at time 0 are available from the dynamic slice.
for d in list({col.rsplit('_t', 1)[0] for col in dynamic_cols}):
    evidence[(d, 0)] = int(test_instance[(d, 0)])

# Query for default_indicator at time period 1.
query_result = infer.query(variables=[('default_indicator', 1)], evidence=evidence)
print("\nInference result for default_indicator at time period 1 given evidence at time period 0:")
print(query_result)

ValueError: invalid literal for int() with base 10: 'C'

In [17]:
import os
import pandas as pd
import numpy as np
from itertools import product
from pgmpy.factors.discrete import TabularCPD

# Create a directory to store the CPD CSV files
output_dir = "cpd_tables"
os.makedirs(output_dir, exist_ok=True)

def save_cpd_as_csv(cpd):
    """
    Convert a TabularCPD into a pandas DataFrame and save it as CSV.
    This function handles both cases: CPDs with evidence (conditional) and without evidence.
    """
    # Try to get evidence-related info (use empty list if not found).
    evidence = getattr(cpd, 'evidence', [])
    evidence_card = getattr(cpd, 'evidence_card', [])
    
    # We'll also use the cpd.state_names if available.
    state_names = getattr(cpd, 'state_names', None)
    
    if not evidence:
        # No evidence: CPD is supposed to be a column vector.
        if state_names and cpd.variable in state_names:
            row_labels = state_names[cpd.variable]
        else:
            row_labels = [f"{cpd.variable}_state_{i}" for i in range(cpd.variable_card)]
        # Get the values and ensure they have shape (variable_card, 1)
        values = np.array(cpd.values)
        if values.ndim == 2 and values.shape[1] != 1:
            # Take only the first column.
            values = values[:, 0:1]
        df = pd.DataFrame(values, index=row_labels, columns=["Probability"])
    else:
        # With evidence: the columns correspond to evidence state combinations.
        evidence_state_lists = []
        for ev in evidence:
            if state_names and ev in state_names:
                evidence_state_lists.append(state_names[ev])
            else:
                idx = evidence.index(ev)
                evidence_state_lists.append([f"{ev}_state_{i}" for i in range(evidence_card[idx])])
        
        # Build a multi-index using the Cartesian product of evidence states.
        col_labels = list(product(*evidence_state_lists))
        if state_names and cpd.variable in state_names:
            row_labels = state_names[cpd.variable]
        else:
            row_labels = [f"{cpd.variable}_state_{i}" for i in range(cpd.variable_card)]
        df = pd.DataFrame(cpd.values, index=row_labels, columns=pd.MultiIndex.from_tuples(col_labels, names=evidence))
    
    filename = os.path.join(output_dir, f"cpd_{cpd.variable}.csv")
    df.to_csv(filename)
    print(f"Saved CPD for {cpd.variable} to {filename}")

def save_cpd_as_text(cpd):
    """Write the string representation of the CPD to a text file."""
    filename = os.path.join(output_dir, f"cpd_{cpd.variable}.txt")
    with open(filename, "w") as f:
        f.write(str(cpd))
    print(f"Saved CPD for {cpd.variable} as text to {filename}")

# Example usage:
# After fitting your model, iterate over CPDs and save them.
for cpd in model.get_cpds():
    file_name = str(cpd.variable) + ".csv"
    cpd.to_csv(filename=file_name)

